# Задание 1

Retention – один из самых важных показателей в компании. Ваша задача – написать функцию, которая будет считать retention игроков (по дням от даты регистрации игрока). Данные лежат в папке shared и имеют следующую структуру:

    shared/problem1-reg_data.csv – данные о времени регистрации
    shared/problem1-auth_data.csv – данные о времени захода пользователей в игру
Функция должна быть написана на python. В ходе решения можно тестировать работу функции как на полном датасете, так и на части (сэмпле) данных.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.stats import ttest_ind, norm

# Загрузка с правильным разделителем
reg_df = pd.read_csv('shared/problem1-reg_data.csv', sep=';')
auth_df = pd.read_csv('shared/problem1-auth_data.csv', sep=';')
reg_df

FileNotFoundError: [Errno 2] No such file or directory: 'shared/problem1-reg_data.csv'

In [ ]:
auth_df

In [ ]:
# Преобразуем дату

def ts_to_date(ts):
    return datetime.utcfromtimestamp(ts).date()

reg_df['reg_date'] = reg_df['reg_ts'].apply(ts_to_date)
auth_df['auth_date'] = auth_df['auth_ts'].apply(ts_to_date)

reg_df

In [ ]:
auth_df

Предварительный анализ данных (EDA) - проверяем качество данных: пропуски, дубликаты, временные рамки.

In [ ]:
# Пропуски
print("\nПропуски в reg_df:")
print(reg_df.isnull().sum())
print("\nПропуски в auth_df:")
print(auth_df.isnull().sum())

# Дубликаты строк
print("\nДубликаты строк в reg_df:", reg_df.duplicated().sum())
print("Дубликаты строк в auth_df:", auth_df.duplicated().sum())

# Дубликаты uid в регистрациях (один пользователь должен регистрироваться один раз)
print("Дубликаты uid в reg_df:", reg_df['uid'].duplicated().sum())





# Диапазон дат
reg_df['year'] = pd.to_datetime(reg_df['reg_date']).dt.year
reg_df['year'].value_counts().sort_index().plot(kind='bar', figsize=(12,5))
plt.title('Количество регистраций по годам')
plt.xlabel('Год')
plt.ylabel('Число пользователей')
plt.show()

def ts_to_date(ts):
    return datetime.utcfromtimestamp(ts).date()

reg_df['reg_date'] = reg_df['reg_ts'].apply(ts_to_date)
auth_df['auth_date'] = auth_df['auth_ts'].apply(ts_to_date)

print("\nМинимальная и максимальная дата регистрации:")
print(reg_df['reg_date'].min(), "→", reg_df['reg_date'].max())

print("Минимальная и максимальная дата авторизации:")
print(auth_df['auth_date'].min(), "→", auth_df['auth_date'].max())

In [ ]:
print("Уникальных пользователей в регистрациях:", reg_df['uid'].nunique())
print("Уникальных пользователей в авторизациях:", auth_df['uid'].nunique())

In [ ]:
reg_uids = set(reg_df['uid'])
auth_uids = set(auth_df['uid'])

print("Пользователей только в регистрациях (без авторизаций):", len(reg_uids - auth_uids))
print("Пользователей в авторизациях, но отсутствующих в регистрациях:", len(auth_uids - reg_uids))

Проверка логики: дата авторизации не раньше даты регистрации. Объединим таблицы и посчитаем разницу в днях. Если встречаются отрицательные значения, значит, в данных есть ошибки.

In [16]:
# Соединяем по uid
merged_check = reg_df[['uid', 'reg_date']].merge(
    auth_df[['uid', 'auth_date']],
    on='uid',
    how='inner'
)

# Разница в днях: auth_date - reg_date
merged_check['diff_days'] = (merged_check['auth_date'] - merged_check['reg_date']).dt.days

print("Минимальная разница (auth - reg):", merged_check['diff_days'].min())
print("Количество строк, где разница отрицательная:", (merged_check['diff_days'] < 0).sum())

NameError: name 'reg_df' is not defined

Активность пользователей: сколько в среднем авторизаций на одного пользователя

In [15]:
auth_counts = auth_df.groupby('uid')['auth_date'].count()
print(auth_counts.describe())

NameError: name 'auth_df' is not defined

In [14]:
plt.figure(figsize=(10,5))
plt.hist(auth_counts, bins=50, log=True, edgecolor='black')
plt.title('Распределение числа авторизаций на пользователя')
plt.xlabel('Число авторизаций')
plt.ylabel('Число пользователей (log)')
plt.show()

NameError: name 'auth_counts' is not defined

<Figure size 720x360 with 0 Axes>

Напишем функцию, которая считает retention отдельно для каждой даты регистрации (когорты).
Функция принимает:

    reg_data – DataFrame с колонками uid, reg_date;

    auth_data – DataFrame с колонками uid, auth_date;

    days_list – список дней N (например, [1, 3, 7, 30]);

    start_date, end_date – необязательные границы дат регистрации (если нужно ограничить период).

Возвращает таблицу (DataFrame), где строки – даты регистрации (когорты), столбцы – дни N, а значения – доля вернувшихся пользователей.

In [13]:
def calculate_retention_cohorts(reg_data, auth_data, days_list, start_date=None, end_date=None):
    """
    Считает retention для каждой когорты (дня регистрации).

    Параметры:
        reg_data (DataFrame): колонки 'uid', 'reg_date'
        auth_data (DataFrame): колонки 'uid', 'auth_date'
        days_list (list): список дней N (например, [1, 3, 7, 30])
        start_date (str или date): начальная дата регистрации (включительно), можно не указывать
        end_date (str или date): конечная дата регистрации (включительно), можно не указывать

    Возвращает:
        DataFrame: индекс = дата регистрации, колонки = дни N, значения = retention (0..1)
    """
    # Переводим start_date и end_date в datetime.date, если они заданы строками
    if start_date is not None:
        start_date = pd.to_datetime(start_date).date()
    if end_date is not None:
        end_date = pd.to_datetime(end_date).date()

    # Фильтруем регистрации по диапазону дат
    if start_date is not None:
        reg_data = reg_data[reg_data['reg_date'] >= start_date]
    if end_date is not None:
        reg_data = reg_data[reg_data['reg_date'] <= end_date]

    # Получаем список уникальных дат регистрации (когорт)
    cohort_dates = sorted(reg_data['reg_date'].unique())

    # Создаём пустой DataFrame для результатов
    result = pd.DataFrame(index=cohort_dates, columns=days_list)
    result.index.name = 'reg_date'

    # Для каждой когорты считаем retention
    for cohort_date in cohort_dates:
        # Пользователи этой когорты
        cohort_users = reg_data[reg_data['reg_date'] == cohort_date]['uid'].unique()
        total_users = len(cohort_users)

        if total_users == 0:
            continue

        # Для каждого дня N находим, сколько пользователей вернулось
        for day in days_list:
            # Дата, в которую нужно проверить возврат
            target_date = cohort_date + timedelta(days=day)

            # Ищем авторизации этих пользователей в target_date
            # Для этого отбираем auth_data по uid из когорты и по нужной дате
            returned_users = auth_data[
                (auth_data['uid'].isin(cohort_users)) &
                (auth_data['auth_date'] == target_date)
            ]['uid'].nunique()

            # Retention = вернувшиеся / общее число
            retention = returned_users / total_users
            result.loc[cohort_date, day] = round(retention, 4)

    return result

In [12]:
retention_sep = calculate_retention_cohorts(
    reg_df, auth_df,
    days_list=[1, 3, 7, 30],
    start_date='2020-09-17',
    end_date='2020-09-23'
)

print(retention_sep)

NameError: name 'calculate_retention_cohorts' is not defined

# Задание 2

Имеются результаты A/B теста, в котором двум группам пользователей предлагались различные наборы акционных предложений. Известно, что ARPU в тестовой группе выше на 5%, чем в контрольной. При этом в контрольной группе 1928 игроков из 202103 оказались платящими, а в тестовой – 1805 из 202667.

Какой набор предложений можно считать лучшим? Какие метрики стоит проанализировать для принятия правильного решения и как?

Формат данных:
user_id	revenue	testgroup
1	0	b
2	0	a
3	0	a
4	0	b
5	0	b

In [11]:
df = pd.read_csv('Проект_1_Задание_2.csv', sep=';')
df

FileNotFoundError: [Errno 2] No such file or directory: 'Проект_1_Задание_2.csv'

In [6]:
# Предварительный анализ
# Платящие
df['is_payer'] = (df['revenue'] > 0).astype(int)

# Группировка
group_stats = df.groupby('testgroup').agg(
    total_users = ('user_id', 'count'),
    payers = ('is_payer', 'sum'),
    total_revenue = ('revenue', 'sum')
).reset_index()

group_stats['conversion'] = group_stats['payers'] / group_stats['total_users']
group_stats['ARPU'] = group_stats['total_revenue'] / group_stats['total_users']
group_stats['ARPPU'] = group_stats['total_revenue'] / group_stats['payers']

group_stats

NameError: name 'df' is not defined

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# --- Верхний ряд: гистограммы выручки ---
for i, group in enumerate(['a', 'b']):
    subset = df[df['testgroup'] == group]['revenue']
    color = '#1f77b4' if group == 'a' else '#ff7f0e'
    axes[0, i].hist(subset, bins=100, color=color, alpha=0.7, label=f'Группа {group.upper()}')
    axes[0, i].set_title(f'Распределение выручки, группа {group.upper()}')
    axes[0, i].set_xlabel('Выручка')
    axes[0, i].set_ylabel('Число пользователей')
    axes[0, i].legend()

# --- Нижний ряд: ящики с усами для платящих ---
for i, group in enumerate(['a', 'b']):
    payers_rev = df[(df['testgroup'] == group) & (df['revenue'] > 0)]['revenue']
    color = '#1f77b4' if group == 'a' else '#ff7f0e'
    bp = axes[1, i].boxplot(payers_rev, vert=True, patch_artist=True,
                            boxprops=dict(facecolor=color, alpha=0.6),
                            medianprops=dict(color='black'))
    axes[1, i].set_title(f'Выручка платящих, группа {group.upper()}')
    axes[1, i].set_ylabel('Выручка')
    axes[1, i].grid(axis='y', linestyle='--', alpha=0.5)

# Убираем пустые третьи панели
axes[0, 2].set_visible(False)
axes[1, 2].set_visible(False)

plt.tight_layout()
plt.show()


Распределение выручки около нуля, т.е. платящих пользоватлей крайне мало. 
Ящик с усами для платящих демонстрирует огромные выбросы у группы А, боксплот не видно из-за масштаба выбросов. 

   

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Платящие
payers = df[df['revenue'] > 0].copy()
payers['Группа'] = payers['testgroup'].map({'a': 'Контрольная группа', 'b': 'Тестовая группа'})

means = payers.groupby('Группа')['revenue'].mean()
colors = ['#2196F3', '#FF9800']

plt.figure(figsize=(12, 6))

sns.histplot(
    data=payers,
    x='revenue',
    hue='Группа',
    element='step',
    palette=colors,
    linewidth=2,
    alpha=0.8
)

ax = plt.gca()
ymax = ax.get_ylim()[1]  # максимальная высота гистограммы

# Разные высоты для плашек, чтобы не пересекались
heights = [0.9 * ymax, 0.7 * ymax]  # первая выше, вторая ниже

for i, (group, mean_val) in enumerate(means.items()):
    plt.axvline(mean_val, color=colors[i], linestyle='--', linewidth=2, alpha=0.8)
    plt.text(mean_val * 1.05, heights[i], f'Средний: {mean_val:,.0f} ₽',
             color=colors[i], fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8),
             ha='left', va='center')

ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.xaxis.get_major_formatter().set_scientific(False)

plt.title('Распределение платежей и средний чек по группам', fontsize=14, pad=15)
plt.xlabel('Размер платежа (руб.)', fontsize=11)
plt.ylabel('Количество игроков', fontsize=11)

ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

В контрольной группе (А) виден отчётливый пик в районе 300–400 рублей — большое число игроков совершают небольшие покупки. В тестовой группе (B) этот пик выражен слабее около 3 000 рублей — там сконцентрирована основная масса платящих. В контрольной группе наблюдается небольшой, но заметный пик экстремально крупных платежей (около 37 000 рублей). В тестовой группе подобных «китов» нет. Несмотря на  разницу в форме распределений, средние очень близки друг к другу. График наглядно демонстрирует, что структура платящей аудитории в группах принципиально разная:

    Группа А: много мелких плательщиков + единичные сверхкрупные.

    Группа B: устойчивое ядро крупных плательщиков без выбросов.
    
Тестовый набор акций меняет структуру монетизации — вместо большого числа мелких платежей мы получаем устойчивый поток средних.

 Для принятия правильного решения стоит проанализировать:
 1. Конверсия в платящего - доля пользователей, которые совершили хотя бы одну покупку. Повышение конверсии означает, что акция мотивирует игроков совершать покупки. Снижение конверсии может сигнализировать, что тестовый набор отпугивает игроков.
 2. ARPU (средняя выручка на пользователя, в т.ч. пользователи без покупок). Если ARPU тестовой группы статистически значимо выше, значит, в среднем каждый пользователь приносит больше денег благодаря тестовым наборам. ARPU может расти и за счет повышения доли платящих и за счет повышения среднего чека. 
 3. ARPPU (средняя выручка на платящего пользователя, пользователи без покупок не учитываются). Стат. значимый рост ARPPU  говорит о том, что тестовый набор стимулирует более дорогие покупки. 

Эти три показателя связаны: ARPU = Конверсия × ARPPU

In [ ]:
# Разбиваем на контрольную (a) и тестовую (b) группы

control = df[df['testgroup'] == 'a']
test = df[df['testgroup'] == 'b']

N_control = len(control)
N_test = len(test)
N_control


In [7]:
N_test

NameError: name 'N_test' is not defined

In [ ]:
# Конверсия
payers_control = (control['revenue'] > 0).sum()
payers_test = (test['revenue'] > 0).sum()

conv_control = payers_control / N_control
conv_test = payers_test / N_test
conv_control

In [ ]:
conv_test

In [ ]:
payers_control

In [ ]:
payers_test

In [ ]:
# ARPU (средняя выручка на пользователя)
arpu_control = control['revenue'].mean()
arpu_test = test['revenue'].mean()
arpu_control

In [ ]:
arpu_test

In [ ]:
# ARPPU (средняя выручка на платящего)
arppu_control = control[control['revenue'] > 0]['revenue'].mean()
arppu_test = test[test['revenue'] > 0]['revenue'].mean()
arppu_control

In [ ]:
arppu_test

In [8]:

print(f"\nКонверсия контроль: {conv_control:.2%}")
print(f"Конверсия тест:    {conv_test:.2%}")
print(f"ARPU контроль:     {arpu_control:.2f}")
print(f"ARPU тест:         {arpu_test:.2f}")
print(f"ARPPU контроль:    {arppu_control:.2f}")
print(f"ARPPU тест:        {arppu_test:.2f}")


NameError: name 'conv_control' is not defined

Тестовое предложение принесло больше денег в расчёте на каждого пользователя (ARPU вырос на 5.3%), но при этом сократило долю платящих (конверсия ниже на 6%). Платящих пользователей стало меньше, но они стали платить больше – ARPPU больше почти на 13%.
Прежду чем делать вывод о том, какой набор предложений можно считать лучшим, нужно провести тест стат. значимости. Для всех статистических тестов в рамках данного исследования установим стандартный уровень значимости α=0,05.

Для конверсии: данные бинарные, t-тест некорректен -он для непрерывных величин. 
Для сравнения конверсии (качественным бинарным признаком: «купил / не купил») при большом объеме выборки (около 202 тысяч наблюдений в каждой группе) лучше всего подходит Z-тест.
Условия Z-теста:
 1. Сравнение долей (пропорций) 
 2. Сравнение средних величин 
 3. Независимость наблюдений

Нулевая гипотеза (H₀): Конверсия в платящего пользователя в контрольной группе равна конверсии в тестовой группе.

Альтернативная гипотеза (H₁): Конверсии в двух группах различаются.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest


# 4. Формируем списки для функции
successes = [payers_control, payers_test]
nobs      = [N_control, N_test]

# 5. Вызываем Z-тест для разницы пропорций
z_stat, p_value = proportions_ztest(count=successes, nobs=nobs)
p_value

Разница в конверсиях статистически значима (p < 0.05)

Распределение ARPU вряд ли будет нормальным: обычно большинство пользователей не платит, небольшое количество пользователей платит.  Считается, что в случае нормального распределения ЦПТ начинает действовать быстрее (т.е. требует меньшего размера выборки, чтобы получилась эта закономерность), но с ненормальными в конечном итоге это также сработает.Выборка большая, поэтому t-тест может дать приемлемый p-value для средней разницы.
условия t-критерия Стьюдента:
 1. Независимость наблюдений
 2. Отсутствие аномальных наблюдений (выбросов)
 3. Равенство дисперсий между ГС
 4. Нормальность обеих ГС
 
Проверим нормальность распределения выборок

In [9]:
# Извлечение выручки всех пользователей (для ARPU)
revenue_control_all = control['revenue']
revenue_test_all = test['revenue']

# Извлечение выручки только платящих (для ARPPU)
revenue_control_payers = control[control['revenue'] > 0]['revenue']
revenue_test_payers = test[test['revenue'] > 0]['revenue']

NameError: name 'control' is not defined

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt

# QQ-plot для контрольной группы (все пользователи)
plt.figure()
sm.qqplot(revenue_control_all, line='s')
plt.title('QQ-plot: ARPU Контроль')
plt.show()

# QQ-plot для тестовой группы (все пользователи)
plt.figure()
sm.qqplot(revenue_test_all, line='s')
plt.title('QQ-plot: ARPU Тест')
plt.show()

Из графиков видно, что распределение не нормально

In [ ]:
# QQ-plot для контрольной группы (платящие)
plt.figure()
sm.qqplot(revenue_control_payers, line='s')
plt.title('QQ-plot: ARPPU Контроль')
plt.show()

# QQ-plot для тестовой группы (платящие)
plt.figure()
sm.qqplot(revenue_test_payers, line='s')
plt.title('QQ-plot: ARPPU Тест')
plt.show()

График для контрольной группы показывает не нормальное распределение, график для тестовой группы показывает отклонения от прямой

In [ ]:
# Гистограмма для контрольной группы (все пользователи)
plt.figure()
plt.hist(revenue_control_all, bins=30, edgecolor='black')
plt.title('Гистограмма ARPU: Контроль')
plt.xlabel('Выручка')
plt.ylabel('Количество')
plt.xlim(0, 5000)  # уберите или измените при необходимости
plt.show()

# Гистограмма для тестовой группы (все пользователи)
plt.figure()
plt.hist(revenue_test_all, bins=30, edgecolor='black')
plt.title('Гистограмма ARPU: Тест')
plt.xlabel('Выручка')
plt.ylabel('Количество')
plt.xlim(0, 5000)
plt.show()

Распределение не нормально

In [ ]:
# Гистограмма для контрольной группы (платящие)
plt.figure()
plt.hist(revenue_control_payers, bins=40, edgecolor='black')
plt.title('Гистограмма ARPPU: Контроль')
plt.xlabel('Выручка')
plt.show()

# Гистограмма для тестовой группы (платящие)
plt.figure()
plt.hist(revenue_test_payers, bins=40, edgecolor='black')
plt.title('Гистограмма ARPPU: Тест')
plt.xlabel('Выручка')
plt.show()

Распределение контрольной группы не нормально, тестовой отдаленно похоже на нормальное, но нормальным не является. 

T-тест устойчив к нарушениям нормальности при больших выборках
При столь больших выборках тест Левена почти всегда будет отвергать нулевую гипотезу о равенстве дисперсий из-за гигантской мощности, поэтому используем correction=True, чтобы всегда использовать вариант Уэлча.

T-тест для ARPU (средняя выручка на пользователя)

    Нулевая гипотеза (H₀):
    Средний ARPU в контрольной группе равен среднему ARPU в тестовой группе.

    Альтернативная гипотеза (H₁):
    Средние ARPU в двух группах различаются.

In [10]:
import pingouin as pg

ttest_arpu = pg.ttest(revenue_test_all, revenue_control_all, correction=True)
ttest_arpu



NameError: name 'revenue_test_all' is not defined

Разница в ARPU НЕ значима p >= 0.05

T-тест для ARPPU (средняя выручка на платящего)

    Нулевая гипотеза (H₀):
    Средний ARPPU в контрольной группе равен среднему ARPPU в тестовой группе.

    Альтернативная гипотеза (H₁):
    Средние ARPPU в двух группах различаются.


In [ ]:
ttest_arppu = pg.ttest(revenue_test_payers, revenue_control_payers, correction=True)
ttest_arppu

/opt/conda/lib/python3.9/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


Разница в ARPPU НЕ значима p >= 0.05

Таким образом, конверсия статистически значимо снизилась в тестовой группе.

Различие ARPU не является статистически значимым.

Различие ARPPU не является статистически значимым.


На основе проведённого анализа лучшим следует считать контрольный набор предложений (a):
Если тестовая группа получила более дорогие предложения, это может объяснить рост ARPPU, но снижение конверсии говорит, что пользователи стали реже совершать покупки.
Тестовый набор не привёл к статистически значимому увеличению ARPU или ARPPU. Тестовый набор статистически значимо снизил конверсию. 
Следовательно, тестовый набор не доказал своей эффективности. 


# Задание 3

В игре Plants & Gardens каждый месяц проводятся тематические события, ограниченные по времени. В них игроки могут получить уникальные предметы для сада и персонажей, дополнительные монеты или бонусы. Для получения награды требуется пройти ряд уровней за определенное время. С помощью каких метрик можно оценить результаты последнего прошедшего события?

Предположим, в другом событии мы усложнили механику событий так, что при каждой неудачной попытке выполнения уровня игрок будет откатываться на несколько уровней назад. Изменится ли набор метрик оценки результата? Если да, то как?

Результаты последнего прошедшего события можно оценить с помощью метрик:

    1. Доля активных пользователей, начавших событие (запустили первый уровень), насколько событие интересно игрокам? 
    2. Среднее количество пройденных уровней, игроки вовлечены в событие? 
    3. Доля игроков полностью завершивших событие, насколько игроки мотивированы дойти до конца? 
    4. Конверсия между этапами: старт - середина - финиш, насколько событие интересно игрокам? 
    5. Среднее время, потраченное на прохождение всего события, долго ли проходить событие?
    6. ARPU / ARPPU участников + сравнение с прошлыми периодами, событие влияет на прибыль?
    7. Процент участников, совершивших хотя бы одну покупку в рамках события, событие мотивирует на покупку? 
    8. Доля игроков, совершивших свою самую первую покупку именно во время нового события, событие стимулирует новых игроков к покупкам? 
    9. Retention, повляело ли событие на удержание? 
    10. Доля игроков, переставших заходить в игру до окончания события, раздрпажает ли игроков событие? 
    11. Определение уровня, на котором игроки массово забросили событие, какой уровень самый сложный? Оправдана ли его сложность? 
    12. Количество использованных бонусов для прохождения события, нужно ли стимулирвоать игркоов к покупке платных бонусов? 
    13. Среднее время, проведенное в игре, вовлекает ли игроков событие?
    14. Количество входа в игру в период времени, повышает ли интерес игроков событие? 
    15. Стоимость потраченных бонусов, событие приносит больше прибыли чем стандартный режим? 
    16. Общее количество игроков во время события, событие привлекает игроков? 
    17. Сколько уровней игроки прошли в среднем, игроки забрасывают событие в начале или ближе к концу? 
   

После усложнения механики событий в набор метрик можно также включить новые метрики:

1. Доля попыток, завершившихся откатом, насколько игра сложная? Сложность оптимальная или завышенная? 
2. Среднее количество уровней, на которое отбрасывается игрок, как сильно игроков отбрасывает назад? На 1 уровень или от финала к началу? 
3. Среднее, медиана и максимум откатов за событие, как часто игроков отбрасывает? 
4. Уровни, на которых откаты происходят чаще всего, какаие уровни самые сложные? Они в конце или в середине? 
5. Доля игроков, столкнувшихся хотя бы с одним откатом, механика вообще влияет на игру?
6. Среднее время, необходимое игроку, чтобы вернуться на уровень, с которого его отбросило, игроки быстро восстаналивают прогресс? Высока ли цена поражения? 
7. Количество игроков, переставших заходить в событие/всю игру, может быть сложная мехника раздражает игроков? 
8. Покупка бонусов, непосредсвтенно влияющих на механику откатов, насколько они востребованы? 
9. Время до повторной попытки после отката, игроки делают паузу или пробуют сразу?

North Star метрика — долгосрочное удержание (retention).

Целевые метрики — Доля игроков полностью завершивших событие и ARPU+ARPPU.

Прокси метрики - конверсия, средний прогресс, доля стартовавших.

Guardrail метрики - отток во время события, общая аудитория.